# Telugu Tokenizer Training (Hugging Face Tokenizers)

**Model**: Hugging Face BPE (Rust-backed)  
**Vocab size**: 16,000  
**Training data**: 100% of cleaned corpus (telugu_cleaned_corpus.jsonl, **400M words**) streamed in batches  
**Output**: `../tokenizer/telugu_tokenizer.json`  

---

### Vocab Size: 16,000

**Why 16K?**
- Balances embedding layer parameter footprint and sequence length inflation.
- Large enough to represent common root morphemes in agglutinative Telugu.
- Prevents extreme sequence length expansion during transformer pretraining.

### Why Hugging Face `tokenizers` library?
- Trains directly from a Python **iterator/generator** in parallelized Rust.
- Memory-efficient: streams the 400M words corpus line-by-line in batches, keeping RAM footprint under 100MB.
- Standard format (`tokenizer.json`) compatible natively with Hugging Face `transformers` pretraining scripts.
- Uses `Metaspace` pre-tokenization and `byte_fallback=True` to match SentencePiece properties.


## 1. Setup

In [7]:
import json, time, re
from pathlib import Path
import tokenizers
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Metaspace

CLEANED       = Path("../datasets/cleaned/telugu_cleaned_corpus.jsonl")
TOKENIZER_DIR = Path("../tokenizer")
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
OUT_MODEL     = TOKENIZER_DIR / "telugu_tokenizer.json"
OUT_VOCAB     = TOKENIZER_DIR / "telugu_tokenizer.vocab"
VOCAB_SIZE    = 16000

# Pristine monolingual alphabet constraint: strip all foreign script characters
ALLOWED_RE    = re.compile(r"[^\u0C00-\u0C7F\u0000-\u007F\u2000-\u200D\u2010-\u201F\s]")

print(f"HuggingFace tokenizers version: {tokenizers.__version__}")
print(f"Input corpus:                 {CLEANED}")
print(f"Output model:                 {OUT_MODEL}")
print(f"Output vocab:                 {OUT_VOCAB}")
print(f"Vocab size:                   {VOCAB_SIZE}")


HuggingFace tokenizers version: 0.23.1
Input corpus:                 ../datasets/cleaned/telugu_cleaned_corpus.jsonl
Output model:                 ../tokenizer/telugu_tokenizer.json
Output vocab:                 ../tokenizer/telugu_tokenizer.vocab
Vocab size:                   16000


## 2. Streaming Batch Iterator

Streams documents from the cleaned JSONL file memory-efficiently to avoid OOM crashes.
Applies regex script-filtering to keep the BPE vocabulary 100% clean of foreign characters.

In [8]:
def batch_iterator(batch_size=10000):
    batch = []
    with open(CLEANED, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                data = json.loads(line)
                text = data.get("text", "").strip()
                # Clean out foreign characters
                text = ALLOWED_RE.sub("", text)
                if text:
                    batch.append(text)
                    if len(batch) == batch_size:
                        yield batch
                        batch = []
            except Exception:
                pass
    if batch:
        yield batch

print("Batch iterator defined.")


Batch iterator defined.


## 3. Train BPE Tokenizer

Configures:
- BPE model with `byte_fallback=True`.
- `Metaspace` pre-tokenizer to handle whitespace as part of tokens.
- Special tokens `[PAD]`, `[UNK]`, `[BOS]`, `[EOS]` corresponding to IDs 0, 1, 2, 3.

In [9]:
# Initialize BPE model with byte fallback
tokenizer = Tokenizer(BPE(unk_token="[UNK]", byte_fallback=True))
tokenizer.pre_tokenizer = Metaspace()

# Configure trainer
trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"],
    show_progress=True
)

print("Starting training from streaming iterator...")
t0 = time.time()
tokenizer.train_from_iterator(batch_iterator(), trainer=trainer)
print(f"Training complete in {time.time() - t0:.1f} seconds.")

# Save tokenizer
tokenizer.save(str(OUT_MODEL))
print(f"Saved tokenizer config JSON to {OUT_MODEL}")

# Save vocabulary as a sorted text file: token 	 id
vocab = tokenizer.get_vocab()
sorted_vocab = sorted(vocab.items(), key=lambda x: x[1])
with open(OUT_VOCAB, "w", encoding="utf-8") as f:
    for token, token_id in sorted_vocab:
        f.write(f"{token}\t{token_id}\n")
print(f"Saved vocabulary text file to {OUT_VOCAB}")

# Verification: Print first 20 tokens
print("\nFirst 20 vocabulary entries:")
for token, token_id in sorted_vocab[:20]:
    print(f"  ID {token_id:<5} : {token!r}")


Starting training from streaming iterator...



Training complete in 2346.4 seconds.
Saved tokenizer config JSON to ../tokenizer/telugu_tokenizer.json
Saved vocabulary text file to ../tokenizer/telugu_tokenizer.vocab

First 20 vocabulary entries:
  ID 0     : '[PAD]'
  ID 1     : '[UNK]'
  ID 2     : '[BOS]'
  ID 3     : '[EOS]'
  ID 4     : '\x01'
  ID 5     : '\x02'
  ID 6     : '\x03'
  ID 7     : '\x04'
  ID 8     : '\x05'
  ID 9     : '\x06'
  ID 10    : '\x07'
  ID 11    : '\t'
  ID 12    : '\n'
  ID 13    : '\x0b'
  ID 14    : '\x0c'
  ID 15    : '\x0e'
  ID 16    : '\x0f'
  ID 17    : '\x11'
  ID 18    : '\x12'
  ID 19    : '\x13'


## 4. Fertility Rate & Statistics

In [10]:
# Load trained tokenizer
tokenizer = Tokenizer.from_file(str(OUT_MODEL))
print(f"Trained vocabulary size: {tokenizer.get_vocab_size():,}")

# Load samples from cleaned corpus
samples_m, samples_d = [], []
with open(CLEANED, encoding="utf-8") as f:
    for line in f:
        if len(samples_m) >= 1000 and len(samples_d) >= 1000:
            break
        try:
            data = json.loads(line)
            text = data.get("text", "")[:500]
            dtype = data.get("type", "manual")
            source = data.get("source", "")
            
            # Nepali source distinction
            is_man = False
            if dtype == "manual":
                is_man = True
            elif "indiccorp" not in source and "cc100" not in source and "wikipedia" not in source:
                is_man = True
                
            if is_man and len(samples_m) < 1000:
                samples_m.append(text)
            elif not is_man and len(samples_d) < 1000:
                samples_d.append(text)
        except Exception:
            continue

def compute_fertility(texts, label):
    total_tokens = total_words = 0
    for text in texts:
        tokens = tokenizer.encode(text).tokens
        total_tokens += len(tokens)
        total_words += len(text.split())
        
    fertility = total_tokens / max(1, total_words)
    print(f"  {label}:")
    print(f"    Fertility (tokens/word): {fertility:.3f}")
    if fertility < 1.5:
        print("    ⚠️ Fertility very low — vocab may be too large")
    elif fertility <= 3.0:
        print("    ✅ Good fertility rate")
    else:
        print("    ⚠️ High fertility — consider increasing vocab size")
    return fertility

print("\n=== Fertility Analysis ===")
compute_fertility(samples_m, "Manual sources")
if samples_d:
    compute_fertility(samples_d, "Downloaded sources")
    compute_fertility(samples_m + samples_d, "Combined")

# Examples
print("\n=== Tokenization Examples ===")
for text in (samples_m[:3] if samples_m else []):
    short = text[:120]
    encoding = tokenizer.encode(short)
    print(f"  Text:   {short}")
    print(f"  Tokens: {encoding.tokens}")
    print(f"  IDs:    {encoding.ids}")
    print()


Trained vocabulary size: 16,000

=== Fertility Analysis ===
  Manual sources:
    Fertility (tokens/word): 2.476
    ✅ Good fertility rate
  Downloaded sources:
    Fertility (tokens/word): 1.633
    ✅ Good fertility rate
  Combined:
    Fertility (tokens/word): 2.132
    ✅ Good fertility rate

=== Tokenization Examples ===
  Text:   వెకొడ౦0 [౬
(షవ%గవ౫ఐ 6 [113కలక్‌

గటివాంత[) ౫ఎ 6౧౧వ టి

06-0. ౮౭౧0౭౧4౭61 1412) 00160600. [1911266 0/ 662౧601
  Tokens: ['▁వె', 'కొ', 'డ', '౦', '0', '▁[', '౬', '\n(', 'ష', 'వ', '%', 'గవ', '౫', 'ఐ', '▁6', '▁[', '11', '3', 'కల', 'క్\u200c', '\n\n', 'గ', 'టి', 'వా', 'ంత', '[', ')', '▁౫', 'ఎ', '▁6', '౧', '౧', 'వ', '▁టి', '\n\n', '0', '6', '-0', '.', '▁', '౮', '౭', '౧', '0', '౭', '౧', '4', '౭', '61', '▁14', '12', ')', '▁', '00', '16', '0', '6', '00', '.', '▁[', '19', '11', '26', '6', '▁0', '/', '▁6', '62', '౧', '60', '1']
  IDs:    [411, 459, 157, 215, 43, 2229, 221, 2648, 179, 177, 32, 7450, 220, 140, 1015, 2229, 2527, 46, 2245, 767, 417, 147, 324, 337, 315, 86,

## 5. Vocabulary Frequency Analysis

In [11]:
from collections import Counter

token_freq = Counter()
for text in samples_m[:500] + samples_d[:500]:
    token_freq.update(tokenizer.encode(text).tokens)

print("Top 30 most frequent tokens in vocabulary:")
for tok, count in token_freq.most_common(30):
    print(f"  {tok:<25} {count:>6}")

# Special token validation
print("\nSpecial Tokens Mapping:")
for tok in ["[PAD]", "[UNK]", "[BOS]", "[EOS]"]:
    print(f"  {tok:<10} ID: {tokenizer.token_to_id(tok)}")


Top 30 most frequent tokens in vocabulary:
  
                           1033
  ను                           689
  న                            608
  ని                           522
  .                            519
  ము                           502
  ,                            499
  ▁ఈ                           444
  గ                            416
  త                            415
  ల                            386
  

                           379
  కు                           360
  క                            345
  మున                          336
  ▁ఆ                           330
  ▁న                           324
  వ                            322
  వి                           321
  లు                           311
  ం                            305
  కి                           305
  ▁                            289
  ▁నా                          285
  ప                            284
  ట                            274
  రు                           273
  ▁'        